In [ ]:
# Install Unsloth and dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# Import libraries
import os
import torch
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, TextStreamer
from unsloth import FastLanguageModel, is_bfloat16_supported

In [ ]:
# Login to Hugging Face (required for gated models)
from huggingface_hub import login
login()  # Enter your token when prompted

## Step 1: Load Model and Tokenizer

In [ ]:
# Configuration
max_seq_length = 2048
model_name = "meta-llama/Meta-Llama-3.1-8B"

# Load model with Unsloth (uses LoRA, not QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,  # Set to True for QLoRA (saves VRAM)
    dtype=None,  # Auto-detect
)

print(f"✓ Model loaded: {model_name}")
print(f"✓ Max sequence length: {max_seq_length}")

## Step 2: Configure LoRA

In [ ]:
# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # LoRA rank (increase for better quality)
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "up_proj", "down_proj", "o_proj", "gate_proj"
    ],
)

print("✓ LoRA configured")
print(f"  - Rank: 32")
print(f"  - Target modules: All linear layers")
print(f"  - Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 3: Prepare Dataset

In [ ]:
# Load datasets
dataset1 = load_dataset("mlabonne/llmtwin", split="train")
dataset2 = load_dataset("mlabonne/FineTome-Alpaca-100k", split="train[:10000]")

# Concatenate datasets
dataset = concatenate_datasets([dataset1, dataset2])

print(f"✓ Dataset loaded")
print(f"  - llmtwin samples: {len(dataset1):,}")
print(f"  - FineTome samples: {len(dataset2):,}")
print(f"  - Total samples: {len(dataset):,}")

In [ ]:
# Format with Alpaca template
alpaca_template = """Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_samples(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_template.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(
    format_samples,
    batched=True,
    remove_columns=dataset.column_names
)

# Split into train/test
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print("✓ Dataset formatted")
print(f"  - Train samples: {len(dataset['train']):,}")
print(f"  - Test samples: {len(dataset['test']):,}")
print("\nSample:")
print(dataset['train'][0]['text'][:300] + "...")

## Step 4: Train Model

In [ ]:
# Training configuration
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        # Learning rate settings
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        warmup_steps=10,
        
        # Training settings
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # Effective batch size = 16
        num_train_epochs=3,
        
        # Optimization
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        
        # Logging and saving
        logging_steps=10,
        eval_steps=100,
        save_steps=500,
        output_dir="output",
        
        # Experiment tracking
        report_to="none",  # Change to "comet_ml" or "wandb" if configured
        seed=42,
    ),
)

print("✓ Trainer configured")
print("\nStarting training...")

In [ ]:
# Train the model
trainer.train()

## Step 5: Test the Model

In [ ]:
# Enable fast inference
FastLanguageModel.for_inference(model)

# Test prompt
instruction = "Write a paragraph to introduce supervised fine-tuning."
message = alpaca_template.format(instruction, "")
inputs = tokenizer([message], return_tensors="pt").to("cuda")

# Generate with streaming
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)

## Step 6: Save and Push to Hub

In [ ]:
# Save locally
model.save_pretrained_merged(
    "model",
    tokenizer,
    save_method="merged_16bit"
)
print("✓ Model saved locally to 'model/' directory")

In [ ]:
# Push to Hugging Face Hub (update with your username)
hub_model_id = "YOUR_USERNAME/TwinLlama-3.1-8B"

model.push_to_hub_merged(
    hub_model_id,
    tokenizer,
    save_method="merged_16bit",
    token=True  # Uses HF_TOKEN from login
)
print(f"✓ Model pushed to https://huggingface.co/{hub_model_id}")

## Optional: Save LoRA Adapters Only

In [ ]:
# Save only LoRA adapters (much smaller)
model.save_pretrained_merged(
    "lora_model",
    tokenizer,
    save_method="lora"
)
print("✓ LoRA adapters saved to 'lora_model/' directory")